# NP-confined PS Analysis — Quasi-2D System

Analysis of binary polymer (PS) blends with nanoparticles in a **quasi-2D slab geometry**
(Lz ≈ 2–3 NP diameters, Lx = Ly >> Lz).

All analyses use **in-plane (xy) quantities** to respect the slab geometry:
- S(k) → in-plane S(k_xy) via z-averaged density + 2-D FFT
- g(r) → in-plane g_2D(r_xy) with PBC in x,y only
- NP–NP RDF → in-plane g_2D(r_xy)
- Cluster analysis → 3-D connected components; sizes reported as xy footprint area
- Interfacial area → 2-D gradient on z-averaged density field (→ interfacial length L_int)
- **New**: z-density profile ρ(z) to visualise NP layering

## Parameters — fill in once files are available
- **phi_np** : TODO
- **RAD**    : [2.5, 5.0]  (NP radius; diameter = 5 or 10)
- **POT**    : TODO
- **Nb**     : TODO
- **rep**    : TODO


In [ ]:
# Cell 1: Imports and Simulation Parameters
import numpy as np
import matplotlib.pyplot as plt
import math
import os
import re
import pandas as pd
import glob
import pickle
from pathlib import Path
from multiprocessing import Pool
from collections import defaultdict
from scipy.optimize import curve_fit

plt.rc('font', family='serif')

# ── Simulation parameters ─────────────────────────────────────────────────────
# TODO: update once files arrive
max_steps = 2_000_001   # total simulation steps
n_frames  = 34          # number of logarithmic output frames
freq      = 1000        # first output frequency

# Box geometry is read per-file from the binary header (Nx and L are anisotropic).
# L[0]=L[1]=Lxy (large), L[2]=Lz (small); similarly Nx[0]=Nx[1]=Nxy, Nx[2]=Nz_small.
# dk = 2*pi / Lxy  (in-plane Fourier resolution — set after reading first file)

# ── File locations ─────────────────────────────────────────────────────────────
# TODO: update to actual directories once files are downloaded
DENSITY_DIR = "/home/gabriel/Projects/TODO_quasi2D/Results/Density"
POS_DIR     = "/home/gabriel/Projects/TODO_quasi2D/Results/Positions"

# TODO: replace PLACEHOLDER with the actual filename prefix
# e.g. "quasi2D_PS_in_NP_frac" or whatever prefix the new files use
FILE_PREFIX = "PLACEHOLDER"

density_pattern = re.compile(
    rf"^{re.escape(FILE_PREFIX)}"
    r"_([\d.]+)_rad_([\d.]+)_POT_([\d.]+)_Nb_(\d+)_rep_(\d+)_densities\.bin$"
)
pos_pattern = re.compile(
    rf"^{re.escape(FILE_PREFIX)}"
    r"_([\d.]+)_rad_([\d.]+)_POT_([\d.]+)_Nb_(\d+)_rep_(\d+)_positions\.bin$"
)

# ── Cache ──────────────────────────────────────────────────────────────────────
CACHE_DIR = Path("/home/gabriel/Projects/TODO_quasi2D/Results/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RECOMPUTE = False

def _cache_path(prefix, phi_np, rad, POT, Nb, rep, init='standard'):
    return CACHE_DIR / f"{prefix}_phi{phi_np}_rad{rad}_POT{POT}_Nb{Nb}_rep{rep}_{init}.pkl"

def _load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def _save_pkl(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


In [ ]:
# Cell 2: Helper Functions (quasi-2D adapted)

from scipy.signal import argrelextrema
from scipy.ndimage import label as _scipy_label


# ── Utilities ──────────────────────────────────────────────────────────────────

def search_files(directory):
    file_list = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_list.append(os.path.join(root, file))
    return file_list


def calculate_log_steps(max_steps, n_frames, freq):
    steps = []
    mult = (max_steps / freq) ** (1.0 / n_frames)
    for _ in range(n_frames):
        freq = freq * mult
        steps.append(int(freq))
    return steps


def choose_n_workers(n_files):
    if n_files % 4 == 0:
        return 4
    elif n_files % 3 == 0:
        return 3
    return 2


# ── Binary readers (unchanged from 3-D notebook) ──────────────────────────────

def read_density_fields(filepath):
    """Read last frame of a density binary. Returns Nx, L, ntypes, rho dict."""
    with open(filepath, 'rb') as f:
        Dim    = np.frombuffer(f.read(4),       dtype=np.int32)[0]
        Nx     = np.frombuffer(f.read(4 * Dim), dtype=np.int32).copy()
        L      = np.frombuffer(f.read(4 * Dim), dtype=np.float32).copy()
        ntypes = np.frombuffer(f.read(4),       dtype=np.int32)[0]
        M      = int(np.prod(Nx))
        fsize  = f.seek(0, 2) or f.tell()
        header = 4 + 4*Dim + 4*Dim + 4
        n_fr   = (fsize - header) // (4 * M * ntypes)
        f.seek(header + (n_fr - 1) * 4 * M * ntypes)
        raw = np.frombuffer(f.read(4 * M * ntypes), dtype=np.float32)
    shape = tuple(Nx[::-1])
    return Nx, L, ntypes, {t: raw[t*M:(t+1)*M].reshape(shape).copy() for t in range(ntypes)}


def read_density_all_frames(filepath):
    """Read all frames of a density binary. Returns Nx, L, ntypes, list of rho dicts."""
    with open(filepath, 'rb') as f:
        Dim    = np.frombuffer(f.read(4),       dtype=np.int32)[0]
        Nx     = np.frombuffer(f.read(4 * Dim), dtype=np.int32).copy()
        L      = np.frombuffer(f.read(4 * Dim), dtype=np.float32).copy()
        ntypes = np.frombuffer(f.read(4),       dtype=np.int32)[0]
        M      = int(np.prod(Nx))
        data   = f.read()
    frame_size = M * ntypes
    n_frames   = len(data) // (4 * frame_size)
    all_raw    = np.frombuffer(data, dtype=np.float32)
    shape      = tuple(Nx[::-1])
    rho_frames = []
    for fr in range(n_frames):
        off = fr * frame_size
        rho_frames.append({t: all_raw[off + t*M : off + (t+1)*M].reshape(shape).copy()
                           for t in range(ntypes)})
    return Nx, L, ntypes, rho_frames


def read_positions_bin(filepath):
    """Read a positions.bin file. Returns ns, Dim, L, tp, molID, frames."""
    with open(filepath, 'rb') as f:
        ns    = np.frombuffer(f.read(4),    dtype=np.int32)[0]
        Dim   = np.frombuffer(f.read(4),    dtype=np.int32)[0]
        L_all = np.frombuffer(f.read(12),   dtype=np.float32)
        tp    = np.frombuffer(f.read(4*ns), dtype=np.int32).copy()
        molID = np.frombuffer(f.read(4*ns), dtype=np.int32).copy()
        raw_frames = []
        fb = 4 * ns * Dim
        while True:
            raw = f.read(fb)
            if len(raw) < fb:
                break
            raw_frames.append(np.frombuffer(raw, dtype=np.float32).reshape(ns, Dim))
    return ns, Dim, L_all[:Dim].copy(), tp, molID, np.array(raw_frames)


# ── Structure factor: in-plane S(k_xy) ────────────────────────────────────────

def bin_sk(k_vals, sk_vals, L_inplane, k_max=2.0):
    """Radially shell-average S(k_xy). L_inplane sets dk = 2*pi/L_inplane."""
    dk          = 2.0 * np.pi / L_inplane
    bin_edges   = np.arange(0.0, k_max + dk, dk)
    bin_centers = bin_edges[:-1] + dk / 2.0
    k  = np.asarray(k_vals,  dtype=np.float64)
    sk = np.asarray(sk_vals, dtype=np.float64)
    counts,  _ = np.histogram(k, bins=bin_edges)
    sk_sum,  _ = np.histogram(k, bins=bin_edges, weights=sk)
    sk2_sum, _ = np.histogram(k, bins=bin_edges, weights=sk**2)
    valid   = counts > 0
    sk_mean = sk_sum[valid] / counts[valid]
    sk_var  = sk2_sum[valid] / counts[valid] - sk_mean**2
    return bin_centers[valid], sk_mean, np.sqrt(np.maximum(sk_var, 0.0)), counts[valid]


def calculate_sk_inplane_from_binary(filepath, type_indices=None, k_max=2.0, max_steps=None, freq=None):
    """
    In-plane structure factor S(k_xy) from a binary density file.

    The 3-D density is averaged over z, then a 2-D FFT is taken:
        S(k_xy) = |FFT2D(rho_z_avg) / M_2d|
    This is equivalent to picking the k_z=0 plane of the 3-D FFT (i.e. the
    Fourier modes that do not vary in z), which is the physically appropriate
    quantity for a quasi-2D slab with a much shorter z-axis.
    """
    PI = np.pi
    if type_indices is None:
        type_indices = [0, 1]

    with open(filepath, 'rb') as f:
        Dim    = np.frombuffer(f.read(4),       dtype=np.int32)[0]
        Nx     = np.frombuffer(f.read(4 * Dim), dtype=np.int32).copy()
        L      = np.frombuffer(f.read(4 * Dim), dtype=np.float32).copy()
        ntypes = np.frombuffer(f.read(4),       dtype=np.int32)[0]

        M      = int(np.prod(Nx))
        shape3 = tuple(Nx[::-1])   # (Nz, Ny, Nx)

        Nx_in  = int(Nx[0])
        Ny_in  = int(Nx[1])
        M_2d   = Nx_in * Ny_in

        dx_in = float(L[0]) / Nx_in
        dy_in = float(L[1]) / Ny_in
        kx_1d = np.fft.fftfreq(Nx_in, d=dx_in) * 2.0 * PI
        ky_1d = np.fft.fftfreq(Ny_in, d=dy_in) * 2.0 * PI
        ky_2d, kx_2d = np.meshgrid(ky_1d, kx_1d, indexing='ij')
        k_xy   = np.sqrt(kx_2d**2 + ky_2d**2).ravel().astype(np.float32)
        k_mask = (k_xy > 0) & (k_xy <= k_max)
        k_out  = k_xy[k_mask]

        bytes_per_frame = 4 * M * ntypes

        header_end = f.tell()
        f.seek(0, 2)
        n_file_frames = (f.tell() - header_end) // bytes_per_frame
        f.seek(header_end)

        file_steps = (calculate_log_steps(max_steps, n_file_frames, freq)
                      if (max_steps is not None and freq is not None and n_file_frames > 0)
                      else None)

        rows      = []
        frame_idx = 0
        while True:
            raw = f.read(bytes_per_frame)
            if len(raw) < bytes_per_frame:
                break
            all_rho = np.frombuffer(raw, dtype=np.float32)
            for t in type_indices:
                if t < 0 or t >= ntypes:
                    continue
                rho_3d    = all_rho[t*M:(t+1)*M].reshape(shape3)
                rho_z_avg = rho_3d.mean(axis=0)          # (Ny, Nx)
                fft2d     = np.fft.fft2(rho_z_avg) / M_2d
                sk_2d     = (np.abs(fft2d)**2).ravel().astype(np.float32)
                row = {
                    'k':                k_out,
                    'structure_factor': sk_2d[k_mask],
                    'frame':            np.int16(frame_idx),
                    'type':             np.int8(t),
                }
                if file_steps is not None:
                    row['step'] = np.int32(file_steps[frame_idx])
                rows.append(pd.DataFrame(row))
            frame_idx += 1

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


# ── Radial density profiles — in-plane ────────────────────────────────────────

def compute_radial_density_profile_2d(frame, L, tp, Nx, np_type=2, poly_type=0,
                                       r_max=None):
    """
    In-plane radial density profile g_2D(r_xy) of poly_type around np_type.

    Uses only x,y components of positions with PBC in x,y.
    Normalisation: 2-D annular area × 2-D bulk area-density (n_poly / Lx*Ly).
    g_2D → 1 at large r_xy.
    """
    Lx, Ly = float(L[0]), float(L[1])
    if r_max is None:
        r_max = min(Lx, Ly) / 2.0
    dr_bin  = min(Lx, Ly) / float(max(int(Nx[0]), int(Nx[1])))
    n_bins  = int(np.ceil(r_max / dr_bin))
    r_edges = np.linspace(0.0, r_max, n_bins + 1)
    r_mid   = 0.5 * (r_edges[1:] + r_edges[:-1])
    counts  = np.zeros(n_bins, dtype=np.float64)

    mask_np   = (tp == np_type)
    mask_poly = (tp == poly_type)
    n_NP      = int(np.sum(mask_np))
    n_poly    = int(np.sum(mask_poly))
    if n_NP == 0 or n_poly == 0:
        return r_mid, np.zeros(n_bins)

    rho_2d_bulk = n_poly / (Lx * Ly)
    np_pos   = frame[mask_np][:, :2].astype(np.float64)
    poly_pos = frame[mask_poly][:, :2].astype(np.float64)

    for i in range(n_NP):
        dr = poly_pos - np_pos[i]
        dr[:, 0] -= Lx * np.round(dr[:, 0] / Lx)
        dr[:, 1] -= Ly * np.round(dr[:, 1] / Ly)
        r = np.sqrt(dr[:, 0]**2 + dr[:, 1]**2)
        hist, _ = np.histogram(r, bins=r_edges)
        counts += hist

    shell_areas = np.pi * (r_edges[1:]**2 - r_edges[:-1]**2)
    gr = counts / (n_NP * rho_2d_bulk * shell_areas)
    return r_mid, gr


def compute_np_np_rdf_2d(frame, L, tp, Nx, np_type=2, r_max=None):
    """
    In-plane NP-NP radial distribution function g_2D(r_xy).
    Pairs counted symmetrically (both i→j and j→i).
    """
    Lx, Ly = float(L[0]), float(L[1])
    if r_max is None:
        r_max = min(Lx, Ly) / 2.0
    dr_bin  = min(Lx, Ly) / float(max(int(Nx[0]), int(Nx[1])))
    n_bins  = int(np.ceil(r_max / dr_bin))
    r_edges = np.linspace(0.0, r_max, n_bins + 1)
    r_mid   = 0.5 * (r_edges[1:] + r_edges[:-1])

    mask_np = (tp == np_type)
    n_NP    = int(np.sum(mask_np))
    if n_NP < 2:
        return r_mid, np.zeros(n_bins)

    np_pos = frame[mask_np][:, :2].astype(np.float64)
    diff   = np_pos[:, np.newaxis, :] - np_pos[np.newaxis, :, :]
    diff[:, :, 0] -= Lx * np.round(diff[:, :, 0] / Lx)
    diff[:, :, 1] -= Ly * np.round(diff[:, :, 1] / Ly)
    r_all  = np.sqrt(diff[:, :, 0]**2 + diff[:, :, 1]**2)
    r_flat = r_all[r_all > 0]

    hist, _ = np.histogram(r_flat, bins=r_edges)
    rho_2d_NP   = (n_NP - 1) / (Lx * Ly)
    shell_areas = np.pi * (r_edges[1:]**2 - r_edges[:-1]**2)
    gr = hist.astype(np.float64) / (n_NP * rho_2d_NP * shell_areas)
    return r_mid, gr


def find_first_minimum(r_mid, gr, r_start=5.0, r_end=12.0, order=2):
    mask  = (r_mid > r_start) & (r_mid < r_end)
    r_sub = r_mid[mask];  g_sub = gr[mask]
    if len(g_sub) < 5:
        return r_start + 2.5
    minima = argrelextrema(g_sub, np.less, order=order)[0]
    return r_sub[minima[0]] if len(minima) > 0 else None


def coordination_number_2d(r_mid, gr, rho_2d_bulk, r_min):
    """
    2-D coordination number: N = rho_2d_bulk * integral_0^r_min g_2D(r) 2*pi*r dr.
    """
    mask = r_mid <= r_min
    r    = r_mid[mask]
    g    = gr[mask]
    return np.trapz(2.0 * np.pi * rho_2d_bulk * g * r, r)


# ── Cluster analysis (3-D; sizes reported as xy footprint area) ───────────────

def cluster_density_field(rho_0, rho_1, target_type=0):
    """
    3-D connected-component clustering with periodic BCs.
    Returns (n_clusters, sizes_array, labels_array).
    """
    binary = (rho_0 > rho_1) if target_type == 0 else (rho_1 > rho_0)
    padded = np.pad(binary, 1, mode='wrap')
    labels_padded, _ = _scipy_label(padded)
    labels = labels_padded[1:-1, 1:-1, 1:-1]
    unique = np.unique(labels)
    unique = unique[unique > 0]
    if len(unique) == 0:
        return 0, np.array([]), labels
    sizes = np.sort(np.array([np.sum(labels == lb) for lb in unique]))[::-1]
    return len(sizes), sizes, labels


# ── Interfacial analysis on z-averaged density ───────────────────────────────

def _iface_grad_2d(rho_0_3d, rho_1_3d, L, Nx):
    """
    Interfacial length L_int via 2-D spectral gradient of z-averaged f_wet.
    Returns scalar L_int in simulation length units.
    """
    Lx, Ly = float(L[0]), float(L[1])
    Nx_in, Ny_in = int(Nx[0]), int(Nx[1])
    dx, dy = Lx / Nx_in, Ly / Ny_in

    kx_1d = np.fft.fftfreq(Nx_in, d=dx) * 2.0 * np.pi
    ky_1d = np.fft.fftfreq(Ny_in, d=dy) * 2.0 * np.pi
    ky_2d, kx_2d = np.meshgrid(ky_1d, kx_1d, indexing='ij')

    eps      = 1e-6
    rho_0_2d = rho_0_3d.mean(axis=0).astype(np.float64)
    rho_1_2d = rho_1_3d.mean(axis=0).astype(np.float64)
    total_2d = rho_0_2d + rho_1_2d
    f_wet    = rho_0_2d / (total_2d + eps)

    F        = np.fft.fft2(f_wet)
    gx       = np.real(np.fft.ifft2(1j * kx_2d * F))
    gy       = np.real(np.fft.ifft2(1j * ky_2d * F))
    grad_mag = np.sqrt(gx**2 + gy**2)

    mask  = total_2d > 0.05
    return float(np.sum(grad_mag * mask) * dx * dy)


def _iface_ip_2d(rho_0_3d, rho_1_3d, L, Nx):
    """
    Interfacial overlap integral on z-averaged density:
      A_ip = integral rho_0_2d * rho_1_2d dA
    Returns scalar in density^2 * area units.
    """
    dx = float(L[0]) / float(Nx[0])
    dy = float(L[1]) / float(Nx[1])
    rho_0_2d = rho_0_3d.mean(axis=0).astype(np.float64)
    rho_1_2d = rho_1_3d.mean(axis=0).astype(np.float64)
    return float(np.sum(rho_0_2d * rho_1_2d) * dx * dy)


# ── z-density profile ─────────────────────────────────────────────────────────

def z_density_profile(rho_dict, L, Nx):
    """
    Compute rho(z) for each particle type by averaging over the xy plane.
    Returns z_centers array and dict {type: rho_z array}.
    """
    Nz = int(Nx[2])
    dz = float(L[2]) / Nz
    z_centers = (np.arange(Nz) + 0.5) * dz
    profiles = {t: rho_dict[t].mean(axis=(1, 2)) for t in rho_dict}
    return z_centers, profiles


In [ ]:
# Cell 3: Load In-Plane Structure Factor Data

def _process_file_sk(file_path):
    fname = os.path.basename(file_path)
    m = density_pattern.match(fname)
    if not m:
        return None
    print(f"  S(k): {fname}")
    df = calculate_sk_inplane_from_binary(file_path, type_indices=[0, 1], k_max=2.0,
                                               max_steps=max_steps, freq=freq)
    if df.empty:
        return None
    df['phi_np'] = m.group(1)
    df['rad']    = m.group(2)
    df['POT']    = m.group(3)
    df['Nb']     = m.group(4)
    df['rep']    = m.group(5)
    df['init']   = 'standard'
    return df

steps = calculate_log_steps(max_steps, n_frames, freq)
print(f"Timesteps (first/last): {steps[0]} … {steps[-1]}")

# Sample first file to print box geometry
_sample_files = search_files(DENSITY_DIR)
for _fp in _sample_files:
    if density_pattern.match(os.path.basename(_fp)):
        _Nx, _L, _nt, _ = read_density_fields(_fp)
        print(f"Box geometry from first file:")
        print(f"  Nx = {_Nx}  (Nxy={_Nx[0]}, Nz={_Nx[2]})")
        print(f"  L  = {_L}  (Lxy={_L[0]:.2f}, Lz={_L[2]:.2f})")
        print(f"  dk_xy = 2*pi/Lxy = {2*np.pi/_L[0]:.4f}")
        break

# Cache-aware dispatch
files = search_files(DENSITY_DIR)
hit_results, miss_files, miss_cps = [], [], []
for fpath in files:
    fname = os.path.basename(fpath)
    m = density_pattern.match(fname)
    if not m:
        continue
    cp = _cache_path('sk', m.group(1), m.group(2), m.group(3), m.group(4), m.group(5))
    if not FORCE_RECOMPUTE and cp.exists():
        _cached = _load_pkl(cp)
        if 'step' in _cached.columns:
            hit_results.append(_cached)
        else:
            miss_files.append(fpath)
            miss_cps.append(cp)
    else:
        miss_files.append(fpath)
        miss_cps.append(cp)

if miss_files:
    n_workers = choose_n_workers(len(miss_files))
    with Pool(n_workers) as pool:
        new_dfs = pool.map(_process_file_sk, miss_files)
    for df_r, cp in zip(new_dfs, miss_cps):
        if df_r is not None:
            _save_pkl(df_r, cp)
            hit_results.append(df_r)

print(f"[cache] {len(hit_results)-len(miss_files)} loaded, {len(miss_files)} computed")
results_sk = [r for r in hit_results if r is not None]
data = pd.concat(results_sk, ignore_index=True)
print(f"Loaded {len(data)} rows from {len(results_sk)} files")


In [ ]:
# Cell 4: Plot S(k_xy) vs k_xy

potentials_of_interest = [str(x) for x in sorted(data['POT'].unique())]
frames_of_interest     = [12, 22, 32]

filtered = data[
    data['POT'].isin(potentials_of_interest) &
    data['frame'].isin(frames_of_interest)
].copy()

frame_colors    = {f: c for f, c in zip(frames_of_interest,
                   __import__('seaborn').color_palette('viridis', len(frames_of_interest)))}
radius_ls       = {'2.5': ':', '5.0': '--'}
poly_labels     = {'0': 'Wetting', '1': 'Non-wetting'}

# Read Lxy from first file for dk
_Nx_ref, _L_ref, _, _ = read_density_fields(search_files(DENSITY_DIR)[0])
Lxy_ref = float(_L_ref[0])

for (phi_np, pot, nb, type_idx), grp in filtered.groupby(['phi_np', 'POT', 'Nb', 'type']):
    plt.figure(figsize=(10, 7))
    for frame in frames_of_interest:
        fdata = grp[grp['frame'] == frame]
        if fdata.empty:
            continue
        t_val = steps[int(frame)] * 0.005
        for rad in sorted(fdata['rad'].unique(), key=float):
            rdata = fdata[fdata['rad'] == rad]
            k_c, sk_m, sk_s, _ = bin_sk(rdata['k'].values, rdata['structure_factor'].values,
                                          Lxy_ref, k_max=2.0)
            ls = radius_ls.get(rad, '-')
            c  = frame_colors[frame]
            plt.plot(k_c, sk_m, ls=ls, color=c, lw=2,
                     label=f't={t_val:.1f}, RAD={rad}')
            plt.fill_between(k_c, sk_m - sk_s, sk_m + sk_s, color=c, alpha=0.12)

    chi = float(pot) / 10.0
    plt.title(rf'$\phi_{{np}}={phi_np}$, $N_b={nb}$, {poly_labels.get(str(type_idx), type_idx)} | '
              rf'$\chi_{{ratio}}={chi:.3f}$', fontsize=16)
    plt.xlabel(r'$|k_{xy}|$', fontsize=20)
    plt.ylabel(r'$S(k_{xy})$', fontsize=20)
    plt.xlim(0.0, 2.0)
    plt.legend(fontsize=10, ncol=2, frameon=True)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


# Radial Density Profiles (In-Plane)

In-plane g_2D(r_xy) of wetting and non-wetting polymer around NP centers.
Only x,y position components are used; PBC applied in x,y only.
Normalisation uses the 2-D bulk area density ρ_2D = N_poly / (Lx·Ly).


In [ ]:
# Cell 6: Compute In-Plane Radial Density Profiles (multiprocessing)

skip = 9   # skip early frames

def _process_position_file(args):
    fpath, fname, phi_np, rad, pot, nb, rep, steps, skip = args

    ns, Dim, L, tp, molID, frames = read_positions_bin(fpath)

    # Try to read Nx from corresponding density file
    dens_subfolder = f"NP_frac_{phi_np}_rad_{rad}_POT_{pot}_Nb_{nb}"
    dens_name = fname.replace("_positions.bin", "_densities.bin")
    dens_path = os.path.join(DENSITY_DIR, dens_subfolder, dens_name)
    if os.path.exists(dens_path):
        with open(dens_path, 'rb') as df_:
            d_Dim  = np.frombuffer(df_.read(4),             dtype=np.int32)[0]
            Nx_arr = np.frombuffer(df_.read(4 * d_Dim),     dtype=np.int32).copy()
    else:
        Nx_arr = np.array([81, 81, 10])   # fallback — update if needed

    file_results = []
    for frame_idx in range(skip, len(frames)):
        for poly_type in [0, 1]:
            r_mid, gr = compute_radial_density_profile_2d(
                frames[frame_idx], L, tp, Nx_arr,
                np_type=2, poly_type=poly_type
            )
            n_poly      = int(np.sum(tp == poly_type))
            rho_2d_bulk = n_poly / (float(L[0]) * float(L[1]))
            file_results.append({
                'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
                'poly_type': poly_type, 'frame': frame_idx,
                'step': steps[frame_idx], 'time': steps[frame_idx] * 0.005,
                'r_mid': r_mid, 'gr': gr, 'rho_2d_bulk': rho_2d_bulk,
                'L': L.copy()
            })
    print(f"  g(r): {fname}")
    return file_results


def _process_npnp_file(args):
    fpath, fname, phi_np, rad, pot, nb, rep, steps, skip = args
    ns, Dim, L, tp, molID, frames = read_positions_bin(fpath)
    dens_path = os.path.join(DENSITY_DIR,
                             f"NP_frac_{phi_np}_rad_{rad}_POT_{pot}_Nb_{nb}",
                             fname.replace("_positions.bin", "_densities.bin"))
    if os.path.exists(dens_path):
        with open(dens_path, 'rb') as df_:
            d_Dim  = np.frombuffer(df_.read(4),         dtype=np.int32)[0]
            Nx_arr = np.frombuffer(df_.read(4 * d_Dim), dtype=np.int32).copy()
    else:
        Nx_arr = np.array([81, 81, 10])

    file_results = []
    for frame_idx in range(skip, len(frames)):
        r_mid, gr = compute_np_np_rdf_2d(frames[frame_idx], L, tp, Nx_arr, np_type=2)
        n_NP      = int(np.sum(tp == 2))
        file_results.append({
            'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
            'frame': frame_idx, 'step': steps[frame_idx],
            'time': steps[frame_idx] * 0.005,
            'r_mid': r_mid, 'gr': gr,
            'rho_2d_NP': n_NP / (float(L[0]) * float(L[1])), 'L': L.copy()
        })
    print(f"  NP-NP: {fname}")
    return file_results


# Build task list
tasks = []
for fname in sorted(os.listdir(POS_DIR)):
    m = pos_pattern.match(fname)
    if not m:
        continue
    phi_np, rad, pot, nb, rep = m.groups()
    tasks.append((os.path.join(POS_DIR, fname), fname,
                  phi_np, rad, pot, nb, rep, steps, skip))

# g(r) — cache-aware
hit_gr, miss_gr, miss_gr_cp = [], [], []
for task in tasks:
    cp = _cache_path('gr', task[2], task[3], task[4], task[5], task[6])
    if not FORCE_RECOMPUTE and cp.exists():
        hit_gr.extend(_load_pkl(cp))
    else:
        miss_gr.append(task); miss_gr_cp.append(cp)

if miss_gr:
    with Pool(choose_n_workers(len(miss_gr))) as pool:
        for res, cp in zip(pool.map(_process_position_file, miss_gr), miss_gr_cp):
            _save_pkl(res, cp); hit_gr.extend(res)

results_gr = hit_gr
print(f"Total g(r) frames: {len(results_gr)}")

# NP-NP RDF — cache-aware
hit_npnp, miss_npnp, miss_npnp_cp = [], [], []
for task in tasks:
    cp = _cache_path('gr_npnp', task[2], task[3], task[4], task[5], task[6])
    if not FORCE_RECOMPUTE and cp.exists():
        hit_npnp.extend(_load_pkl(cp))
    else:
        miss_npnp.append(task); miss_npnp_cp.append(cp)

if miss_npnp:
    with Pool(choose_n_workers(len(miss_npnp))) as pool:
        for res, cp in zip(pool.map(_process_npnp_file, miss_npnp), miss_npnp_cp):
            _save_pkl(res, cp); hit_npnp.extend(res)

results_gr_npnp = hit_npnp
print(f"Total NP-NP RDF frames: {len(results_gr_npnp)}")


In [ ]:
# Cell 7: Plot In-Plane g_2D(r_xy) and NP-NP RDF

import seaborn as sns

poly_labels  = {0: 'Wetting', 1: 'Non-wetting'}
poly_colors  = {0: 'tab:blue', 1: 'tab:orange'}
radius_ls    = {'2.5': ':', '5.0': '--'}
frames_plot  = [10, 20, 30]
frame_alphas = {f: 0.3 + 0.7 * i / (len(frames_plot) - 1) for i, f in enumerate(frames_plot)}

# ── g_2D(r_xy) ────────────────────────────────────────────────────────────────
avg_gr = {}
by_key = defaultdict(list)
for res in results_gr:
    if res['frame'] in frames_plot:
        key = (res['phi_np'], res['Nb'], res['POT'], res['rad'],
               res['poly_type'], res['frame'])
        by_key[key].append(res)

for key, rlist in by_key.items():
    gr_arr = np.array([r['gr'] for r in rlist])
    avg_gr[key] = {'r_mid': rlist[0]['r_mid'], 'gr_mean': gr_arr.mean(0),
                   'gr_std': gr_arr.std(0), 'time': rlist[0]['time']}

for (phi_np, nb, pot) in sorted(set(k[:3] for k in avg_gr)):
    chi = float(pot) / 10.0
    fig, ax = plt.subplots(figsize=(10, 6))
    rads = sorted(set(k[3] for k in avg_gr if k[:3] == (phi_np, nb, pot)), key=float)
    for rad in rads:
        ls = radius_ls.get(rad, '-')
        for poly_type in [0, 1]:
            for fi in frames_plot:
                key = (phi_np, nb, pot, rad, poly_type, fi)
                if key not in avg_gr:
                    continue
                d = avg_gr[key]
                lbl = (rf'{poly_labels[poly_type]}, RAD={rad}, t={d["time"]:.1f}'
                       if fi == frames_plot[0] else None)
                ax.plot(d['r_mid'], d['gr_mean'],
                        color=poly_colors[poly_type], lw=1.5,
                        linestyle=ls, alpha=frame_alphas[fi], label=lbl)
                ax.fill_between(d['r_mid'],
                                d['gr_mean'] - d['gr_std'],
                                d['gr_mean'] + d['gr_std'],
                                color=poly_colors[poly_type],
                                alpha=0.08 * frame_alphas[fi])
    ax.axhline(1.0, color='k', lw=0.8, ls='--', alpha=0.4)
    ax.set_xlabel(r'In-plane distance $r_{xy}$ [$\sigma$]', fontsize=16)
    ax.set_ylabel(r'$g_{2D}(r_{xy})$', fontsize=16)
    ax.set_title(rf'$\phi_{{np}}={phi_np}$, $N_b={nb}$, $\chi_{{ratio}}={chi:.3f}$', fontsize=14)
    ax.legend(fontsize=9, frameon=True)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

# ── NP-NP RDF ─────────────────────────────────────────────────────────────────
pot_colors_np = dict(zip(
    sorted(set(r['POT'] for r in results_gr_npnp), key=float),
    sns.color_palette('viridis', len(set(r['POT'] for r in results_gr_npnp)))
))

for (phi_np, nb) in sorted(set((r['phi_np'], r['Nb']) for r in results_gr_npnp)):
    rads = sorted(set(r['rad'] for r in results_gr_npnp
                      if r['phi_np']==phi_np and r['Nb']==nb), key=float)
    fig, axes = plt.subplots(1, len(rads), figsize=(6*len(rads), 5), squeeze=False)
    for col, rad in enumerate(rads):
        ax  = axes[0][col]
        sub = [r for r in results_gr_npnp
               if r['phi_np']==phi_np and r['Nb']==nb and r['rad']==rad
               and r['frame'] == max(r2['frame'] for r2 in results_gr_npnp)]
        pots_here = sorted(set(r['POT'] for r in sub), key=float)
        for pot in pots_here:
            grp = [r for r in sub if r['POT'] == pot]
            gr_m = np.array([r['gr'] for r in grp]).mean(0)
            ax.plot(grp[0]['r_mid'], gr_m, color=pot_colors_np[pot], lw=1.8,
                    label=rf'$\chi$={float(pot)/10:.2f}')
        contact = 2.0 * float(rad)
        ax.axvline(contact, color='k', lw=1, ls='--', alpha=0.5,
                   label=f'Contact 2r={contact:.1f}')
        ax.axhline(1.0, color='gray', lw=0.8, ls='--', alpha=0.4)
        ax.set_xlabel(r'$r_{xy}$ [$\sigma$]', fontsize=14)
        ax.set_ylabel(r'$g_{2D}^{NP-NP}(r_{xy})$', fontsize=14)
        ax.set_title(f'RAD={rad}', fontsize=13)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    fig.suptitle(rf'NP-NP RDF (in-plane) — $\phi_{{np}}={phi_np}$, $N_b={nb}$', fontsize=14)
    fig.tight_layout()
    plt.show()


In [ ]:
# Cell 8: Coordination Numbers (2-D)

_coord_cache = CACHE_DIR / "coord_all_runs_quasi2d.pkl"
if not FORCE_RECOMPUTE and _coord_cache.exists():
    coord_df = _load_pkl(_coord_cache)
    print(f"[cache] Loaded coordination numbers ({len(coord_df)} rows)")
else:
    wetting_rmin = {}
    for res in results_gr:
        if res['poly_type'] != 0:
            continue
        rad_val = float(res['rad']) + 1.0
        r_min   = find_first_minimum(res['r_mid'], res['gr'],
                                     r_start=rad_val, r_end=12.0, order=1)
        if r_min is None:
            r_min = rad_val + 3.0
        key = (res['phi_np'], res['rad'], res['POT'], res['Nb'], res['rep'], res['frame'])
        wetting_rmin[key] = r_min

    coord_results = []
    for res in results_gr:
        key   = (res['phi_np'], res['rad'], res['POT'], res['Nb'], res['rep'], res['frame'])
        r_min = wetting_rmin.get(key, float(res['rad']) + 4.0)
        N_c   = coordination_number_2d(res['r_mid'], res['gr'],
                                        res['rho_2d_bulk'], r_min)
        coord_results.append({
            'phi_np': res['phi_np'], 'rad': res['rad'], 'POT': res['POT'],
            'Nb': res['Nb'], 'rep': res['rep'], 'poly_type': res['poly_type'],
            'frame': res['frame'], 'step': res['step'], 'time': res['time'],
            'r_min': r_min, 'N_coord': N_c
        })

    coord_df = pd.DataFrame(coord_results)
    coord_df['chi_ratio'] = coord_df['POT'].astype(float) / 10.0
    _save_pkl(coord_df, _coord_cache)
    print(f"[cache] Saved ({len(coord_df)} rows)")

# f_wet contact fraction
coord_total = coord_df.groupby(['phi_np', 'rad', 'POT', 'Nb', 'rep', 'frame', 'time'])
fwet_records = []
for keys, grp in coord_total:
    wet_row = grp[grp['poly_type'] == 0]
    nw_row  = grp[grp['poly_type'] == 1]
    if wet_row.empty or nw_row.empty:
        continue
    n_wet = wet_row['N_coord'].values[0]
    n_nw  = nw_row['N_coord'].values[0]
    total = n_wet + n_nw
    fwet_records.append({**dict(zip(['phi_np','rad','POT','Nb','rep','frame','time'], keys)),
                         'f_wet_contact': n_wet / total if total > 0 else np.nan,
                         'chi_ratio': float(keys[2]) / 10.0})
fwet_df = pd.DataFrame(fwet_records)

import seaborn as sns
pot_list_c  = sorted(coord_df['POT'].unique(), key=float)
pot_cols_c  = sns.color_palette('viridis', len(pot_list_c))
pot_cmap_c  = dict(zip(pot_list_c, pot_cols_c))
rad_ls_c    = {'2.5': ':', '5.0': '--'}

for (phi_np, nb), sub in coord_df.groupby(['phi_np', 'Nb']):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    for ax, poly_type in zip(axes, [0, 1]):
        pt = sub[sub['poly_type'] == poly_type]
        for pot, pot_df in pt.groupby('POT'):
            for rad, r_df in pot_df.groupby('rad'):
                st = r_df.groupby('time')['N_coord'].agg(['mean','std']).reset_index()
                ax.errorbar(st['time'], st['mean'], yerr=st['std'],
                            color=pot_cmap_c[pot], ls=rad_ls_c.get(rad, '-'),
                            marker='o', ms=3, capsize=2,
                            label=rf'$\chi$={float(pot)/10:.2f}, RAD={rad}')
        ax.set_xscale('log')
        ax.set_xlabel('Time', fontsize=13)
        ax.set_ylabel(r'$N_{coord,2D}$', fontsize=13)
        ax.set_title(f'{"Wetting" if poly_type==0 else "Non-wetting"}', fontsize=13)
        ax.legend(fontsize=7, frameon=True)
        ax.grid(alpha=0.3, which='both')
    fig.suptitle(rf'2-D Coordination Number — $\phi_{{np}}={phi_np}$, $N_b={nb}$', fontsize=14)
    fig.tight_layout()
    plt.show()


# Local Density Distributions

P(f_wet) computed on the z-averaged density grid.
f_wet = ρ₀_2D / (ρ₀_2D + ρ₁_2D), restricted to polymer-rich pixels (total > 0.05).


In [ ]:
# Cell 10: Number Fraction Distributions from Z-averaged Density

def _process_density_file(args):
    fpath, phi_np, rad, pot, nb, rep = args
    _, _, _, rho = read_density_fields(fpath)
    rho_wet_2d = rho[0].mean(axis=0).ravel().astype(np.float64)
    rho_nw_2d  = rho[1].mean(axis=0).ravel().astype(np.float64)
    total      = rho_wet_2d + rho_nw_2d
    valid      = total > 0.05
    f_wet      = rho_wet_2d[valid] / total[valid]
    return [{'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
             'f_wet': f_wet}]

dens_tasks = []
for subfolder in sorted(os.listdir(DENSITY_DIR)):
    sub_path = os.path.join(DENSITY_DIR, subfolder)
    if not os.path.isdir(sub_path):
        continue
    for fname in sorted(os.listdir(sub_path)):
        m = density_pattern.match(fname)
        if m:
            dens_tasks.append((os.path.join(sub_path, fname), *m.groups()))

hit_d, miss_d, miss_d_cp = [], [], []
for task in dens_tasks:
    cp = _cache_path('density_frac', task[1], task[2], task[3], task[4], task[5])
    if not FORCE_RECOMPUTE and cp.exists():
        hit_d.extend(_load_pkl(cp))
    else:
        miss_d.append(task); miss_d_cp.append(cp)

if miss_d:
    with Pool(choose_n_workers(len(miss_d))) as pool:
        for batch, cp in zip(pool.map(_process_density_file, miss_d), miss_d_cp):
            _save_pkl(batch, cp); hit_d.extend(batch)

density_results = hit_d
import seaborn as sns

hist_bins   = np.linspace(0.0, 1.0, 81)
bin_centers = 0.5 * (hist_bins[1:] + hist_bins[:-1])

pot_list_d  = sorted(set(r['POT'] for r in density_results), key=float)
pot_cols_d  = sns.color_palette('viridis', len(pot_list_d))
pot_cmap_d  = dict(zip(pot_list_d, pot_cols_d))

grouped_d = defaultdict(lambda: defaultdict(list))
for res in density_results:
    grouped_d[(res['phi_np'], res['Nb'], res['rad'])][res['POT']].append(res['f_wet'])

for (phi_np, nb, rad) in sorted(grouped_d):
    fig, ax = plt.subplots(figsize=(9, 5))
    for pot in pot_list_d:
        rep_arrays = grouped_d[(phi_np, nb, rad)].get(pot)
        if not rep_arrays:
            continue
        rep_hists = np.array([np.histogram(f, bins=hist_bins, density=True)[0]
                               for f in rep_arrays])
        P_mean = rep_hists.mean(0)
        P_std  = rep_hists.std(0)
        chi    = float(pot) / 10.0
        c      = pot_cmap_d[pot]
        ax.plot(bin_centers, P_mean, color=c, lw=1.5,
                label=rf'$\chi_{{ratio}}={chi:.3f}$ (n={len(rep_arrays)})')
        ax.fill_between(bin_centers, P_mean - P_std, P_mean + P_std, color=c, alpha=0.2)
    ax.set_yscale('log')
    ax.set_xlabel(r'Wetting fraction $f_{\mathrm{wet}}$ (z-averaged)', fontsize=15)
    ax.set_ylabel('Probability density', fontsize=15)
    ax.set_title(rf'$\phi_{{np}}={phi_np}$, $N_b={nb}$, RAD={rad}', fontsize=13)
    ax.legend(fontsize=9, frameon=True)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


# Cluster Analysis

3-D connected-component labeling on the full density grid (periodic BCs).
In the quasi-2D slab the clusters are naturally slab-like objects spanning z.
Cluster size is reported as **xy footprint area** (voxel_count × dx × dy),
which is the physically meaningful 2-D extent of each cluster.

Minimum cluster size threshold: 1 polymer chain footprint =
  Nb / (dx × dy × Lz)  [in grid cells].


In [ ]:
# Cell 12: Cluster Statistics (3-D, sizes as xy footprint area)

def _process_cluster_file(args):
    fpath, phi_np, rad, pot, nb, rep, steps, skip = args
    Nx_arr, L_arr, ntypes, rho_frames = read_density_all_frames(fpath)
    dx = float(L_arr[0]) / float(Nx_arr[0])
    dy = float(L_arr[1]) / float(Nx_arr[1])
    dz = float(L_arr[2]) / float(Nx_arr[2])
    voxel_xy = dx * dy          # xy footprint per voxel
    voxel_vol = dx * dy * dz
    Nz_grid   = int(Nx_arr[2])

    # 1 chain occupies Nb sigma^3 total; footprint in grid cells = Nb / (voxel_vol * Nz_grid)
    min_size_voxels = math.ceil(int(nb) / voxel_vol)

    file_results = []
    for fi in range(skip, len(rho_frames)):
        rho  = rho_frames[fi]
        step = steps[fi] if fi < len(steps) else fi
        t    = step * 0.005
        for target_type in [0, 1]:
            _, sizes, _ = cluster_density_field(rho[0], rho[1], target_type)
            sizes = sizes[sizes >= min_size_voxels]
            n_cl  = len(sizes)
            # Convert voxel count to xy area (divide by Nz since clusters span all z)
            sizes_xy = sizes / Nz_grid * voxel_xy   # sigma^2
            file_results.append({
                'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
                'poly_type': target_type, 'frame': fi, 'step': step, 'time': t,
                'n_clusters': n_cl,
                'largest_cluster_area': float(sizes_xy[0]) if n_cl > 0 else 0.0,
                'mean_cluster_area':    float(sizes_xy.mean()) if n_cl > 0 else 0.0,
                'sizes_xy': sizes_xy,
            })
    print(f"  clusters: {os.path.basename(fpath)}")
    return file_results


cluster_tasks = []
for subfolder in sorted(os.listdir(DENSITY_DIR)):
    sub_path = os.path.join(DENSITY_DIR, subfolder)
    if not os.path.isdir(sub_path):
        continue
    for fname in sorted(os.listdir(sub_path)):
        m = density_pattern.match(fname)
        if m:
            cluster_tasks.append((os.path.join(sub_path, fname), *m.groups(), steps, skip))

hit_cl, miss_cl, miss_cl_cp = [], [], []
for task in cluster_tasks:
    cp = _cache_path('cluster', task[1], task[2], task[3], task[4], task[5])
    if not FORCE_RECOMPUTE and cp.exists():
        hit_cl.extend(_load_pkl(cp))
    else:
        miss_cl.append(task); miss_cl_cp.append(cp)

if miss_cl:
    with Pool(choose_n_workers(len(miss_cl))) as pool:
        for res, cp in zip(pool.map(_process_cluster_file, miss_cl), miss_cl_cp):
            _save_pkl(res, cp); hit_cl.extend(res)

cluster_results = hit_cl
cluster_df = pd.DataFrame(cluster_results)
cluster_df['chi_ratio'] = cluster_df['POT'].astype(float) / 10.0
print(f"Total cluster analyses: {len(cluster_df)}")


In [ ]:
# Cell 13: Plot Cluster Statistics

import seaborn as sns

poly_labels = {0: 'Wetting', 1: 'Non-wetting'}
pot_list_cl  = sorted(cluster_df['POT'].unique(), key=float)
pot_cols_cl  = sns.color_palette('viridis', len(pot_list_cl))
pot_cmap_cl  = dict(zip(pot_list_cl, pot_cols_cl))
rads_cl      = sorted(cluster_df['rad'].unique(), key=float)

for (phi_np, nb), sub in cluster_df.groupby(['phi_np', 'Nb']):
    fig, axes = plt.subplots(len(rads_cl), 2,
                             figsize=(14, 5 * len(rads_cl)), sharey=False, sharex=True)
    if len(rads_cl) == 1:
        axes = [axes]
    for row, rad in enumerate(rads_cl):
        rad_df = sub[sub['rad'] == rad]
        for col, poly_type in enumerate([0, 1]):
            ax = axes[row][col]
            for pot, pot_df in rad_df[rad_df['poly_type'] == poly_type].groupby('POT'):
                st = pot_df.groupby('time')['n_clusters'].agg(['mean','std']).reset_index()
                chi = float(pot) / 10.0
                ax.errorbar(st['time'], st['mean'], yerr=st['std'],
                            marker='o', ms=3, color=pot_cmap_cl[pot], capsize=2,
                            label=rf'$\chi$={chi:.2f}')
            ax.set_xscale('log')
            ax.set_xlabel('Time', fontsize=12)
            ax.set_ylabel('# clusters', fontsize=12)
            ax.set_title(f'RAD={rad} — {poly_labels[poly_type]}', fontsize=12)
            ax.legend(fontsize=7); ax.grid(alpha=0.3, which='both')
    fig.suptitle(rf'Number of Clusters — $\phi_{{np}}={phi_np}$, $N_b={nb}$', fontsize=14)
    fig.tight_layout(); plt.show()

# ── Largest cluster xy area ───────────────────────────────────────────────────
for (phi_np, nb), sub in cluster_df.groupby(['phi_np', 'Nb']):
    fig, axes = plt.subplots(len(rads_cl), 2,
                             figsize=(14, 5 * len(rads_cl)), sharey=False, sharex=True)
    if len(rads_cl) == 1:
        axes = [axes]
    for row, rad in enumerate(rads_cl):
        rad_df = sub[sub['rad'] == rad]
        for col, poly_type in enumerate([0, 1]):
            ax = axes[row][col]
            for pot, pot_df in rad_df[rad_df['poly_type'] == poly_type].groupby('POT'):
                st = pot_df.groupby('time')['largest_cluster_area'].agg(['mean','std']).reset_index()
                chi = float(pot) / 10.0
                ax.errorbar(st['time'], st['mean'], yerr=st['std'],
                            marker='s', ms=3, color=pot_cmap_cl[pot], capsize=2,
                            label=rf'$\chi$={chi:.2f}')
            ax.set_xscale('log')
            ax.set_xlabel('Time', fontsize=12)
            ax.set_ylabel(r'Largest cluster xy area [$\sigma^2$]', fontsize=12)
            ax.set_title(f'RAD={rad} — {poly_labels[poly_type]}', fontsize=12)
            ax.legend(fontsize=7); ax.grid(alpha=0.3, which='both')
    fig.suptitle(rf'Largest Cluster — $\phi_{{np}}={phi_np}$, $N_b={nb}$', fontsize=14)
    fig.tight_layout(); plt.show()


# Interfacial Length (In-Plane)

Computed on the z-averaged density field ρ_2D(x, y).

**Gradient method**: L_int = ∫ |∇_xy f_wet| dA  (units: σ)
- f_wet = ρ₀_2D / (ρ₀_2D + ρ₁_2D); spectral (FFT) gradient; polymer-rich pixels only.
- Physically: total length of the wetting/non-wetting interface projected onto xy.

**Inner product method**: A_ip = ∫ ρ₀_2D · ρ₁_2D dA  (units: σ²)
- Maximised in mixed regions, vanishes in fully separated system.


In [ ]:
# Cell 15: Interfacial Length — Gradient Method (compute)

def _process_interface_file(args):
    fpath, phi_np, rad, pot, nb, rep, init, steps = args
    Nx_arr, L_arr, ntypes, rho_frames = read_density_all_frames(fpath)
    file_results = []
    for fi, rho in enumerate(rho_frames):
        step = steps[fi] if fi < len(steps) else fi
        t    = step * 0.005
        L_int = _iface_grad_2d(rho[0], rho[1], L_arr, Nx_arr)
        file_results.append({
            'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
            'init': init, 'frame': fi, 'step': step, 'time': t,
            'L_int': L_int,
        })
    print(f"  L_int: {os.path.basename(fpath)}")
    return file_results


iface_tasks = []
for subfolder in sorted(os.listdir(DENSITY_DIR)):
    sub_path = os.path.join(DENSITY_DIR, subfolder)
    if not os.path.isdir(sub_path):
        continue
    for fname in sorted(os.listdir(sub_path)):
        m = density_pattern.match(fname)
        if m:
            iface_tasks.append((os.path.join(sub_path, fname), *m.groups(), 'standard', steps))

hit_if, miss_if, miss_if_cp = [], [], []
for task in iface_tasks:
    cp = _cache_path('iface_grad', task[1], task[2], task[3], task[4], task[5], task[6])
    if not FORCE_RECOMPUTE and cp.exists():
        hit_if.extend(_load_pkl(cp))
    else:
        miss_if.append(task); miss_if_cp.append(cp)

if miss_if:
    with Pool(choose_n_workers(len(miss_if))) as pool:
        for res, cp in zip(pool.map(_process_interface_file, miss_if), miss_if_cp):
            _save_pkl(res, cp); hit_if.extend(res)

iface_results = hit_if
iface_df = pd.DataFrame(iface_results)
iface_df['chi_ratio'] = iface_df['POT'].astype(float) / 10.0
print(f"Total interfacial length rows: {len(iface_df)}")


In [ ]:
# Cell 16: Interfacial Length — Gradient Method (plot)

import seaborn as sns

pot_list_if  = sorted(iface_df['POT'].unique(), key=float)
pot_cols_if  = sns.color_palette('viridis', len(pot_list_if))
pot_cmap_if  = dict(zip(pot_list_if, pot_cols_if))
rads_if      = sorted(iface_df['rad'].unique(), key=float)

for (phi_np, nb), sub in iface_df.groupby(['phi_np', 'Nb']):
    fig, axes = plt.subplots(1, len(rads_if), figsize=(7*len(rads_if), 5),
                             sharey=True, squeeze=False)
    for col, rad in enumerate(rads_if):
        ax     = axes[0][col]
        rad_df = sub[sub['rad'] == rad]
        for pot in pot_list_if:
            chi   = float(pot) / 10.0
            g     = rad_df[rad_df['POT'] == pot]
            if g.empty: continue
            st = g.groupby('time')['L_int'].agg(['mean','std']).reset_index()
            ax.plot(st['time'], st['mean'], color=pot_cmap_if[pot], lw=1.8,
                    label=rf'$\chi_{{ratio}}={chi:.2f}$')
            ax.fill_between(st['time'],
                            st['mean'] - st['std'], st['mean'] + st['std'],
                            color=pot_cmap_if[pot], alpha=0.15)
        ax.set_xscale('log')
        ax.set_xlabel(r'Time $t$', fontsize=13)
        ax.set_ylabel(r'$L_\mathrm{int}$ [$\sigma$]', fontsize=13)
        ax.set_title(f'RAD={rad}', fontsize=12)
        ax.legend(fontsize=8, frameon=True)
        ax.grid(alpha=0.3, which='both')
    fig.suptitle(rf'Interfacial Length (gradient) — $\phi_{{np}}={phi_np}$, $N_b={nb}$',
                 fontsize=14)
    fig.tight_layout(); plt.show()


In [ ]:
# Cell 17: Interfacial Overlap — Inner Product Method (compute)

def _process_interface_ip_file(args):
    fpath, phi_np, rad, pot, nb, rep, init, steps = args
    Nx_arr, L_arr, ntypes, rho_frames = read_density_all_frames(fpath)
    file_results = []
    for fi, rho in enumerate(rho_frames):
        step = steps[fi] if fi < len(steps) else fi
        t    = step * 0.005
        A_ip = _iface_ip_2d(rho[0], rho[1], L_arr, Nx_arr)
        file_results.append({
            'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
            'init': init, 'frame': fi, 'step': step, 'time': t,
            'A_int_ip': A_ip,
        })
    print(f"  A_ip: {os.path.basename(fpath)}")
    return file_results


iface_ip_tasks = []
for subfolder in sorted(os.listdir(DENSITY_DIR)):
    sub_path = os.path.join(DENSITY_DIR, subfolder)
    if not os.path.isdir(sub_path):
        continue
    for fname in sorted(os.listdir(sub_path)):
        m = density_pattern.match(fname)
        if m:
            iface_ip_tasks.append((os.path.join(sub_path, fname), *m.groups(), 'standard', steps))

hit_ip, miss_ip, miss_ip_cp = [], [], []
for task in iface_ip_tasks:
    cp = _cache_path('iface_ip', task[1], task[2], task[3], task[4], task[5], task[6])
    if not FORCE_RECOMPUTE and cp.exists():
        hit_ip.extend(_load_pkl(cp))
    else:
        miss_ip.append(task); miss_ip_cp.append(cp)

if miss_ip:
    with Pool(choose_n_workers(len(miss_ip))) as pool:
        for res, cp in zip(pool.map(_process_interface_ip_file, miss_ip), miss_ip_cp):
            _save_pkl(res, cp); hit_ip.extend(res)

iface_ip_df = pd.DataFrame(hit_ip)
iface_ip_df['chi_ratio'] = iface_ip_df['POT'].astype(float) / 10.0
print(f"Total inner-product rows: {len(iface_ip_df)}")


In [ ]:
# Cell 18: Interfacial Overlap — Inner Product Method (plot)

import seaborn as sns

pot_list_ip  = sorted(iface_ip_df['POT'].unique(), key=float)
pot_cols_ip  = sns.color_palette('viridis', len(pot_list_ip))
pot_cmap_ip  = dict(zip(pot_list_ip, pot_cols_ip))
rads_ip      = sorted(iface_ip_df['rad'].unique(), key=float)

for (phi_np, nb), sub in iface_ip_df.groupby(['phi_np', 'Nb']):
    fig, axes = plt.subplots(1, len(rads_ip), figsize=(7*len(rads_ip), 5),
                             sharey=True, squeeze=False)
    for col, rad in enumerate(rads_ip):
        ax     = axes[0][col]
        rad_df = sub[sub['rad'] == rad]
        for pot in pot_list_ip:
            chi = float(pot) / 10.0
            g   = rad_df[rad_df['POT'] == pot]
            if g.empty: continue
            st  = g.groupby('time')['A_int_ip'].agg(['mean','std']).reset_index()
            ax.plot(st['time'], st['mean'], color=pot_cmap_ip[pot], lw=1.8,
                    label=rf'$\chi_{{ratio}}={chi:.2f}$')
            ax.fill_between(st['time'],
                            st['mean'] - st['std'], st['mean'] + st['std'],
                            color=pot_cmap_ip[pot], alpha=0.15)
        ax.set_xscale('log')
        ax.set_xlabel(r'Time $t$', fontsize=13)
        ax.set_ylabel(r'$\int \rho_0\,\rho_1\,dA$', fontsize=13)
        ax.set_title(f'RAD={rad}', fontsize=12)
        ax.legend(fontsize=8, frameon=True)
        ax.grid(alpha=0.3, which='both')
    fig.suptitle(rf'Interfacial Overlap (inner product) — $\phi_{{np}}={phi_np}$, $N_b={nb}$',
                 fontsize=14)
    fig.tight_layout(); plt.show()


# z-Structure Analysis (New — Quasi-2D Specific)

ρ(z) = density averaged over the xy plane as a function of z-position.
Reveals NP layering and polymer stratification in the short direction.
Computed from the last frame of each density binary.


In [ ]:
# Cell 20: z-Density Profiles

type_labels  = {0: 'Wetting polymer', 1: 'Non-wetting polymer', 2: 'Nanoparticles'}
type_colors  = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:green'}

def _process_zdensity_file(args):
    fpath, phi_np, rad, pot, nb, rep = args
    Nx_arr, L_arr, ntypes, rho = read_density_fields(fpath)
    z_c, profiles = z_density_profile(rho, L_arr, Nx_arr)
    return {'phi_np': phi_np, 'rad': rad, 'POT': pot, 'Nb': nb, 'rep': rep,
            'z': z_c, 'profiles': profiles, 'Lz': float(L_arr[2])}

z_tasks = []
for subfolder in sorted(os.listdir(DENSITY_DIR)):
    sub_path = os.path.join(DENSITY_DIR, subfolder)
    if not os.path.isdir(sub_path):
        continue
    for fname in sorted(os.listdir(sub_path)):
        m = density_pattern.match(fname)
        if m:
            z_tasks.append((os.path.join(sub_path, fname), *m.groups()))

n_workers_z = choose_n_workers(len(z_tasks))
with Pool(n_workers_z) as pool:
    z_results = pool.map(_process_zdensity_file, z_tasks)

# Group by (phi_np, Nb, RAD, POT) — average across replicas
by_params = defaultdict(list)
for res in z_results:
    by_params[(res['phi_np'], res['Nb'], res['rad'], res['POT'])].append(res)

import seaborn as sns

for (phi_np, nb, rad, pot) in sorted(by_params):
    chi   = float(pot) / 10.0
    rlist = by_params[(phi_np, nb, rad, pot)]
    z_c   = rlist[0]['z']
    ntypes_here = len(rlist[0]['profiles'])

    fig, ax = plt.subplots(figsize=(8, 4))
    for t in range(ntypes_here):
        all_rhoz = np.array([r['profiles'][t] for r in rlist])
        mean_rho = all_rhoz.mean(0)
        std_rho  = all_rhoz.std(0)
        ax.plot(z_c, mean_rho, color=type_colors.get(t, f'C{t}'),
                lw=2, label=type_labels.get(t, f'Type {t}'))
        ax.fill_between(z_c, mean_rho - std_rho, mean_rho + std_rho,
                        color=type_colors.get(t, f'C{t}'), alpha=0.2)

    ax.set_xlabel(r'$z$ [$\sigma$]', fontsize=14)
    ax.set_ylabel(r'$\langle \rho \rangle_{xy}(z)$', fontsize=14)
    ax.set_title(rf'z-profile — $\phi_{{np}}={phi_np}$, $N_b={nb}$, RAD={rad}, '
                 rf'$\chi_{{ratio}}={chi:.3f}$', fontsize=13)
    ax.set_xlim(0, rlist[0]['Lz'])
    ax.legend(fontsize=10, frameon=True)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
